In [3]:
#!pip install transformers datasets torch


In [1]:
# Install necessary libraries (uncomment if not already installed)
# !pip install transformers datasets torch
import wandb
wandb.init(mode="disabled", project="qa_generation")


from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from datasets import load_dataset

# 1. Load the IMDb dataset
dataset = load_dataset('imdb')

# 2. Load the tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Tokenize the dataset
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=256)

tokenized_train = dataset['train'].map(tokenize, batched=True)
tokenized_test = dataset['test'].map(tokenize, batched=True)

# 4. Set the format for PyTorch
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# 5. Define training arguments (only essential parameters)
training_args = TrainingArguments(
    output_dir='./results',      # Required: Output directory
    num_train_epochs=1,          # Optional: Defaults to 3
)

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,

)

# 7. Fine-tune the model
trainer.train()

# 8. Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

# 9. Save the fine-tuned model
model.save_pretrained("./fine_tuned_distilbert_classifier")
tokenizer.save_pretrained("./fine_tuned_distilbert_classifier")

# 10. Inference using pipeline
classifier = pipeline("text-classification", model="./fine_tuned_distilbert_classifier", tokenizer="./fine_tuned_distilbert_classifier")

# Example texts
texts = [
    "I absolutely loved this movie! The performances were outstanding.",
    "This was the worst film I have ever seen. Terrible plot and acting."
]

# Get predictions
predictions = classifier(texts)
for text, pred in zip(texts, predictions):
    print(f"Text: {text}\nPrediction: {pred}\n")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
500,0.431000
1000,0.357800
1500,0.325500
2000,0.300100
2500,0.289300
3000,0.277300


Evaluation results: {'eval_loss': 0.2645466923713684, 'eval_runtime': 99.3131, 'eval_samples_per_second': 251.729, 'eval_steps_per_second': 31.466, 'epoch': 1.0}


Device set to use cuda:0


Text: I absolutely loved this movie! The performances were outstanding.
Prediction: {'label': 'LABEL_1', 'score': 0.9977658987045288}

Text: This was the worst film I have ever seen. Terrible plot and acting.
Prediction: {'label': 'LABEL_0', 'score': 0.9964562058448792}



In [3]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# Load the fine-tuned model and tokenizer
loaded_model = AutoModelForSequenceClassification.from_pretrained("./fine_tuned_distilbert_classifier")
loaded_tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_distilbert_classifier")

# Initialize the pipeline for inference
classifier = pipeline("text-classification", model=loaded_model, tokenizer=loaded_tokenizer)

# Example texts for prediction
texts = [
    "I absolutely loved this movie! The performances were outstanding.",
    "This was the worst film I have ever seen. Terrible plot and acting."
]

# Get predictions
predictions = classifier(texts)
for text, pred in zip(texts, predictions):
    print(f"Text: {text}\nPrediction: {pred}\n")


Device set to use cuda:0


Text: I absolutely loved this movie! The performances were outstanding.
Prediction: {'label': 'LABEL_1', 'score': 0.9977658987045288}

Text: This was the worst film I have ever seen. Terrible plot and acting.
Prediction: {'label': 'LABEL_0', 'score': 0.9964562058448792}



In [5]:
!zip -r classifier.zip ./fine_tuned_distilbert_classifier


'zip' is not recognized as an internal or external command,
operable program or batch file.


In [12]:
#from google.colab import files
#files.download('classifier.zip')

## Understanding Transfer Learning, Heads, and Fine-Tuning
**Transfer Learning**

Transfer learning is a machine learning technique where a model trained on a
 - large dataset (e.g., a general-purpose dataset like Wikipedia) is reused for a different, specific task (e.g., sentiment analysis).
 - The idea is to leverage the pre-trained model's learned knowledge (such as understanding of syntax, grammar, and general concepts) to perform well on a new task, even with limited labeled data.

**Model Heads**

In transformer-based models like DistilBERT, the "head" refers to the final layers that adapt the model for a specific task. For example:

- Pre-trained Head: The original head used during general pre-training (e.g., masked language modeling).

- Task-specific Head: A new head added for downstream tasks like classification, question answering, or named entity recognition.

- This head learns task-specific patterns while relying on the base model's pre-trained knowledge.

**Fine-Tuning vs Transfer Learning**
- Fine-Tuning:
The entire model, including the base (pre-trained layers) and task-specific head, is updated during training.

- Used when you have a sufficient amount of labeled data for the target task.

**Transfer Learning (Feature Extraction)**:

- The pre-trained base model is frozen (its weights are not updated), and only the task-specific head is trained.
-Useful when labeled data is limited or when the pre-trained knowledge is already highly relevant to the new task.

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from datasets import load_dataset

# 1. Load the IMDb dataset
dataset = load_dataset('imdb')

# 2. Load the tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Freeze the base model's layers (for transfer learning)
for param in model.base_model.parameters():
    param.requires_grad = False  # Freeze the layers

# 4. Tokenize the dataset
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=256)

tokenized_train = dataset['train'].map(tokenize, batched=True)
tokenized_test = dataset['test'].map(tokenize, batched=True)

# 5. Set the format for PyTorch
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# 6. Define training arguments (only essential parameters)
training_args = TrainingArguments(
    output_dir='./results',      # Required: Output directory
    num_train_epochs=1,          # Optional: Defaults to 3
)

# 7. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

# 8. Train the classification head
trainer.train()

# 9. Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

# 10. Save the fine-tuned model
model.save_pretrained("./transfer_learning_distilbert_classifier")
tokenizer.save_pretrained("./transfer_learning_distilbert_classifier")

# 11. Inference using pipeline
classifier = pipeline("text-classification", model="./transfer_learning_distilbert_classifier", tokenizer="./transfer_learning_distilbert_classifier")

# Example texts
texts = [
    "I absolutely loved this movie! The performances were outstanding.",
    "This was the worst film I have ever seen. Terrible plot and acting."
]

# Get predictions
predictions = classifier(texts)
for text, pred in zip(texts, predictions):
    print(f"Text: {text}\nPrediction: {pred}\n")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
500,0.596000
1000,0.452900
1500,0.429200
2000,0.412400
2500,0.414100
3000,0.395800


Evaluation results: {'eval_loss': 0.3862665295600891, 'eval_runtime': 98.0872, 'eval_samples_per_second': 254.875, 'eval_steps_per_second': 31.859, 'epoch': 1.0}


Device set to use cuda:0


Text: I absolutely loved this movie! The performances were outstanding.
Prediction: {'label': 'LABEL_1', 'score': 0.9967827796936035}

Text: This was the worst film I have ever seen. Terrible plot and acting.
Prediction: {'label': 'LABEL_0', 'score': 0.9782693982124329}

